In [407]:
import polars as pl
from pathlib import Path
from unidecode import unidecode
from datetime import date
import calendar
BASE_DIR = Path.cwd().parent / 'data_input' / 'nao_consignados'

In [408]:
retorno = pl.read_excel(f'{BASE_DIR}/RETORNO TODOS 06.2026.xlsx')
base_ativos = pl.read_excel(f'{BASE_DIR}/BASE ATIVOS E CANCELADOS 05.xlsx')
coopart = pl.read_excel(f'{BASE_DIR}/ANALITICO COPARTICIPAÇÃO SIGRH - 05.2026 (consignado) COMPLETO.xlsx')
remuneracao = pl.read_excel(f'{BASE_DIR}/REMUNERAÇÃO 05.2026.xlsx')


Could not determine dtype for column 22, falling back to string


In [409]:
def formata_coluna(cols:list) -> list:
    cols_norm = []
    for col in cols:
        cols_norm.append(unidecode(col.lower().replace(' ', '_')).replace('-', ''))

    return cols_norm

retorno.columns = formata_coluna(retorno.columns)
base_ativos.columns = formata_coluna(base_ativos.columns)
remuneracao.columns = formata_coluna(remuneracao.columns)
coopart.columns = formata_coluna(coopart.columns)

retorno =  retorno.select(pl.all().exclude(['mesclado', 'qtde_rubricas']))


### Retorno folha

In [410]:
retorno = retorno.with_columns([
    pl.col('cpf__titular').str.slice(-11)
]).with_columns([
    pl.concat_str([pl.col('cpf__titular'), pl.col('rubrica')], separator='_').alias('chave')
])

### Remuneração 

In [411]:
remuneracao = remuneracao.with_columns([
    ((4 * pl.col('remuneracao_inas')) / 100).round(2).alias('valor_mensalidade'),
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).with_columns([
    pl.when(pl.col('valor_mensalidade') < 578).then(578)
    .when(pl.col('valor_mensalidade') > 1538).then(1538)
    .otherwise('valor_mensalidade').alias('valor_mensalidade')
]).filter(pl.col('orgao') != '911')

### Base ativos

In [412]:
base_ativos1 = base_ativos.with_columns([
    pl.col('cpf_titular').str.replace_all(r'[^\d]', ''),
    pl.col('cpf_segurado').str.replace_all(r'[^\d]', '')
]).select(['cpf_titular', 'cpf_segurado', 'tipo_segurado', 'data_adesao','data_nascimento' , 'codigo_da_empresa', 'matricula_titular_8_digitos', 'idade'])

base_ativos1 = base_ativos1.rename({
    'codigo_da_empresa':'orgao', 'matricula_titular_8_digitos':'matricula'
}).filter(pl.col('orgao') != '911')

base_ativos1 = base_ativos1.with_columns([
    pl.when(pl.col('tipo_segurado') == 'TITULAR').then(pl.lit('41134'))
    .when(pl.col('idade') <= 18).then(pl.lit('41135'))
    .when(pl.col('idade') <= 23).then(pl.lit('41171'))
    .when(pl.col('idade') <= 28).then(pl.lit('41175'))
    .when(pl.col('idade') <= 33).then(pl.lit('41191'))
    .when(pl.col('idade') <= 38).then(pl.lit('41192'))
    .when(pl.col('idade') <= 43).then(pl.lit('41193'))
    .when(pl.col('idade') <= 48).then(pl.lit('41221'))
    .when(pl.col('idade') <= 53).then(pl.lit('41222'))
    .when(pl.col('idade') <= 58).then(pl.lit('41223'))
    .otherwise(pl.lit('41224')).alias('rubrica')
]).with_columns([
    pl.when(pl.col('rubrica') == '41134').then(0)
    .when(pl.col('rubrica') == '41135').then(225)
    .when(pl.col('rubrica') == '41171').then(250)
    .when(pl.col('rubrica') == '41175').then(340)
    .when(pl.col('rubrica') == '41191').then(370)
    .when(pl.col('rubrica') == '41192').then(420)
    .when(pl.col('rubrica') == '41193').then(479)
    .when(pl.col('rubrica') == '41221').then(530)
    .when(pl.col('rubrica') == '41222').then(657)
    .when(pl.col('rubrica') == '41223').then(764)
    .otherwise(853).alias('mensalidade')
])

In [413]:
base_ativos1 = base_ativos1.with_columns([
    pl.concat_str([pl.col('cpf_titular'), pl.col('rubrica')], separator='_').alias('chave')
])
base_ativos1 = base_ativos1.unique(subset=['cpf_titular', 'cpf_segurado'])

In [414]:
base_ativos1 = (base_ativos1.join(
    remuneracao[['cpf', 'valor_mensalidade']], left_on=['cpf_titular'], right_on=['cpf'], how='left')
    ).with_columns([
        pl.col('valor_mensalidade').fill_null(578)
    ])

base_ativos1 = base_ativos1.with_columns([
    pl.when(pl.col('mensalidade') == 0).then('valor_mensalidade').otherwise('mensalidade').alias('mensalidade')
]).drop('valor_mensalidade')


### Pro-rata

In [415]:
mes = date.today().replace(day=1, month=5)
total_dias = calendar.monthrange(mes.year, mes.month)[1]

pro_rata = base_ativos1.filter(pl.col('data_adesao') > mes)

pro_rata = pro_rata.with_columns([
    ((pl.col('mensalidade') / total_dias) * (total_dias - pl.col('data_adesao').dt.day())).round(2).alias('valor_pro_rata')
])

pro_rata = pro_rata.with_columns([
    (pl.lit('5') + pl.col('rubrica').str.slice(1)).alias('rubrica')
]).with_columns([
    pl.concat_str(['cpf_titular', 'rubrica'], separator='_').alias('chave')
])

### Copart

In [416]:
coopart = coopart.with_columns([
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).rename({'copart_mes_titular':'titular', 'copart_mes_dependente':'dependente'})

In [417]:
rubrica_coopart = coopart.unpivot(
    index=['emp_cod', 'matricula', 'cpf'],
    on=['dependente', 'titular'],
    variable_name='tipo_beneficiario',
    value_name='valor'
).with_columns([
    pl.when(pl.col('tipo_beneficiario') == 'dependente')
    .then(pl.lit('41144')).otherwise(pl.lit('41136')).alias('rubrica')
]).with_columns([
    pl.concat_str([pl.col('cpf'), pl.col('rubrica')], separator='_').alias('chave')
]).filter(pl.col('valor') > 0)

### Resultado final

In [460]:
coopart_boleto = rubrica_coopart.join(retorno, on='chave', how='anti')
coopart_boleto = coopart_boleto.pivot(
    index='cpf',
    values='valor',
    on='tipo_beneficiario'
)
coopart_boleto = coopart_boleto.with_columns([
    pl.col('dependente').fill_null(0),
    pl.col('titular').fill_null(0)
]).rename({'dependente':'copartipacao_dependente', 'titular':'copartipacao_titular'})

In [461]:
pro_rata_boleto = pro_rata.join(retorno, on='chave', how='anti')

pro_rata_boleto = pro_rata_boleto.pivot(
    index='cpf_titular',
    values='valor_pro_rata',
    on='tipo_segurado'
)

pro_rata_boleto = pro_rata_boleto.with_columns([
    pl.col('TITULAR').fill_null(0),
    pl.col('DEPENDENTE').fill_null(0)
]).rename({'TITULAR':'pro_rata_titular', 'DEPENDENTE':'pro_rata_dependente', 'cpf_titular':'cpf'})

In [462]:
mensalidade_boletos = base_ativos1.join(retorno, on='chave', how='anti')

mensalidade_boletos = mensalidade_boletos.group_by(['cpf_titular', 'tipo_segurado']).agg(
    pl.col('mensalidade').sum()
)

mensalidade_boletos = mensalidade_boletos.pivot(
    index='cpf_titular',
    values='mensalidade',
    on='tipo_segurado'
)

mensalidade_boletos = mensalidade_boletos.with_columns([
    pl.col('TITULAR').fill_null(0),
    pl.col('DEPENDENTE').fill_null(0)
]).rename({'TITULAR':'mensalidade_titular', 'DEPENDENTE':'mensalidade_dependente', 'cpf_titular':'cpf'})

In [468]:
relatorio_final = (mensalidade_boletos
    .join(coopart_boleto, on='cpf', how='full')
    .with_columns([
        pl.col('cpf').fill_null(pl.col('cpf_right'))
    ]).drop('cpf_right')
    ).fill_null(0)

In [469]:
relatorio_final = (relatorio_final
    .join(pro_rata_boleto, on='cpf', how='full')
    .with_columns([
        pl.col('cpf').fill_null(pl.col('cpf_right'))
    ]).drop('cpf_right')
    ).fill_null(0)

In [471]:
relatorio_final = relatorio_final.with_columns([
    (pl.col('mensalidade_titular') + pl.col('mensalidade_dependente') + pl.col('copartipacao_dependente') + 
     pl.col('copartipacao_titular') + pl.col('pro_rata_titular') + pl.col('pro_rata_dependente') 
     ).alias('total_cobranca')
])

In [472]:
relatorio_final['total_cobranca'].sum()

1564729.2799999998

In [474]:
relatorio_debug = pl.read_excel(f'{BASE_DIR}/Relatorio_NAO_CONSIGNADOS.xlsx')
relatorio_debug.columns = formata_coluna(relatorio_debug.columns)

Could not determine dtype for column 0, falling back to string
Could not determine dtype for column 14, falling back to string
Could not determine dtype for column 15, falling back to string


In [479]:
relatorio_debug = relatorio_debug.select(['cpf_titular', 'mensalidade_titular', 'pro_rata_titular', 'mensalidade_dependente', 'pro_rata_dependente',
                        'coparticipacao_titular', 'coparticipacao_dependente', 'total_cobranca'])